Learning the Ropes F1 Machine Learning

In [21]:
import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sys import prefix
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
 

In [8]:
races = pd.read_csv('../archive/races.csv')
results = pd.read_csv('../archive/results.csv')

dataset = pd.merge(races,results,on='raceId')

dataset = dataset.sort_values(['date','round'],ascending=True)
#display(dataset.sample(5))
dataset['DriverForm'] = dataset.groupby('driverId')['positionOrder'].transform(lambda x: x.shift(1).rolling(window=3).mean())

max_data = dataset[dataset['driverId'] == 830]
display(max_data[['year', 'name', 'positionOrder', 'DriverForm']].tail(10))

,year,name,positionOrder,DriverForm
27039,2025,Dutch Grand Prix,2,6.000000
27058,2025,Italian Grand Prix,1,5.000000
27078,2025,Azerbaijan Grand Prix,1,4.000000
27099,2025,Singapore Grand Prix,2,1.333333
27118,2025,United States Grand Prix,1,1.333333
27140,2025,Mexico City Grand Prix,3,1.333333
27160,2025,São Paulo Grand Prix,3,2.000000
27178,2025,Las Vegas Grand Prix,1,2.333333
27198,2025,Qatar Grand Prix,1,2.333333
27218,2025,Abu Dhabi Grand Prix,1,1.666667


In [19]:

#encoded_dataset = pd.get_dummies(dataset,columns=['constructorId', 'circuitId'],prefix=['Team','Track'])
#print(encoded_dataset.columns.tolist())

dataset['Win'] = (dataset['positionOrder'] == 1).astype(int)
ml_data = pd.get_dummies(dataset,columns=['constructorId', 'circuitId'],prefix=['Team','Track'])

ml_data = ml_data.replace(r'\N', np.nan)
ml_data['grid'] = pd.to_numeric(ml_data['grid'], errors='coerce')
ml_data['DriverForm'] = pd.to_numeric(ml_data['DriverForm'], errors='coerce')
ml_data = ml_data.dropna(subset=['DriverForm', 'grid'])

train_data = ml_data[ml_data['year'] < 2024]

# Test the model on the future (2024 and 2025)
test_data = ml_data[ml_data['year'] >= 2024]

print(f"Training rows: {len(train_data)}")
print(f"Testing rows: {len(test_data)}")


Training rows: 24155
Testing rows: 920


In [23]:
feature_columns = ['grid', 'DriverForm'] + [col for col in ml_data.columns if col.startswith('Team_') or col.startswith('Track_')]

X_train = train_data[feature_columns]
y_train = train_data['Win']

X_test = test_data[feature_columns]
y_test = test_data['Win']

# 2. INITIALIZE THE ALGORITHM
# max_iter gives the algorithm enough time to find the mathematical patterns without timing out
model = LogisticRegression(class_weight='balanced', max_iter=2000)

# 3. TRAIN THE MODEL 
# This is where the actual Machine Learning happens. 
# It looks at decades of history and learns the mathematical weights of starting on pole vs starting 10th.
model.fit(X_train, y_train)

# 4. PREDICT THE FUTURE
# We hand it the 2024/2025 data (X_test) and ask it to guess who won each race
predictions = model.predict(X_test)

# 5. GRADE THE EXAM
# We compare its guesses against the actual real-world results (y_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

print("--- CONFUSION MATRIX ---")
# y_test is the real world, predictions is the model's guesses
matrix = confusion_matrix(y_test, predictions)
print(matrix)

# 2. GENERATE THE REPORT CARD
print("\n--- CLASSIFICATION REPORT ---")
# zero_division=0 just stops a warning if the model NEVER guessed a winner
print(classification_report(y_test, predictions, zero_division=0))

Model Accuracy: 76.41%
--- CONFUSION MATRIX ---
[[657 216]
 [  1  46]]

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       1.00      0.75      0.86       873
           1       0.18      0.98      0.30        47

    accuracy                           0.76       920
   macro avg       0.59      0.87      0.58       920
weighted avg       0.96      0.76      0.83       920



Will a driver finish in top 10?

In [ ]:
# 2. SETUP TARGET
dataset['InPoints'] = (dataset['positionOrder'] < 11).astype(int)
# 3. FIX DATA CONVERSION (Do this BEFORE get_dummies)
# If you run get_dummies first, these columns are deleted and turned into Team_1, Team_2, etc.
dataset['grid'] = pd.to_numeric(dataset['grid'], errors='coerce')
dataset['constructorId'] = pd.to_numeric(dataset['constructorId'], errors='coerce')
# 4. ENCODE CATEGORICAL DATA
# We keep 'grid' as a number (it's better for the model than separate Grid_1, Grid_2 columns)
# We only dummies the Team (constructorId)
predict_data = pd.get_dummies(dataset, columns=['constructorId'], prefix=['Team'])
# 5. CLEAN DATA
predict_data = predict_data.dropna(subset=['DriverForm', 'grid', 'InPoints'])
# 6. SPLIT DATA (Note the variable name consistency!)
train_data_point = predict_data[predict_data['year'] < 2024]
test_data_point = predict_data[predict_data['year'] >= 2024]
print(f"Training rows: {len(train_data_point)}")
print(f"Testing rows: {len(test_data_point)}")
# 7. CHOOSE FEATURES
# We use 'grid' as a numeric column and the 'Team_' dummy columns
feature_columns = ['grid'] + [col for col in predict_data.columns if col.startswith('Team_')]
X_train = train_data_point[feature_columns]
y_train = train_data_point['InPoints']
X_test = test_data_point[feature_columns]
y_test = test_data_point['InPoints']
# 8. TRAIN & PREDICT
model = LogisticRegression(class_weight='balanced', max_iter=2000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
# 9. RESULTS
print(f"Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print("\n--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, predictions))
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, predictions, zero_division=0))

Training rows: 24155
Testing rows: 920
Model Accuracy: 70.76%

--- CONFUSION MATRIX ---
[[274 183]
 [ 86 377]]

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       0.76      0.60      0.67       457
           1       0.67      0.81      0.74       463

    accuracy                           0.71       920
   macro avg       0.72      0.71      0.70       920
weighted avg       0.72      0.71      0.70       920

